# NRPy in Ten Minutes

Follow one small pipeline from a symbolic expression to generated C and a
registered C function.

Navigation: [Index](../index.ipynb) |
Previous: [Getting Started with NRPy](../0-getting_started/install_and_run.ipynb) |
Next: [Parameters](params.ipynb)

## Learning Goals

- See the full NRPy workflow in miniature.
- Connect a symbolic energy expression to generated C code.
- Store a generated C function for later use.

## Words for This Notebook

- **Symbolic expression:** a formula represented by Python objects instead of
  immediate decimal numbers.
- **Grid field:** a quantity sampled at many grid points, like a wave value at
  each point.
- **C function:** a named block of C code that can be called later.

## One Miniature Pipeline

The toy field energy density is

$$
E = \frac{1}{2}\left(v^2 + c^2 u^2\right).
$$

| Step | NRPy object | Evidence to inspect |
| --- | --- | --- |
| Build symbols | `uu`, `vv`, `overview_wave_speed` | printed C assignment |
| Emit C | `energy_density` assignment | field and parameter names |
| Store function | `overview_energy_density` | callable C interface |

## Import SymPy for the Toy Expression

SymPy creates the symbolic expression that NRPy will translate.

In [1]:
import sympy as sp

## Import NRPy Core Tools

These imports expose C code generation, C function registration,
gridfunction registration, and code parameters.

In [2]:
import nrpy.c_codegen as ccg
from nrpy.c_function import CFunction_dict, register_CFunction
import nrpy.grid as grid
import nrpy.params as par

## Clear Tutorial State

Rerunning the notebook should start from the same named objects each time.

In [3]:
removed_functions = []
for name in ["overview_energy_density"]:
    if CFunction_dict.pop(name, None) is not None:
        removed_functions.append(name)

removed_gridfunctions = []
for name in ["uu", "vv"]:
    if grid.glb_gridfcs_dict.pop(name, None) is not None:
        removed_gridfunctions.append(name)

par.glb_code_params_dict.pop("overview_wave_speed", None)
par.set_parval_from_str("Infrastructure", "BHaH")

print("cleared functions:", removed_functions)
print("cleared gridfunctions:", removed_gridfunctions)
print("infrastructure:", par.parval_from_str("Infrastructure"))

cleared functions: []
cleared gridfunctions: []
infrastructure: BHaH


## Build the Symbolic Energy Density

The printed assignment is the first math-to-code bridge. Look for the two
wave variables and the named wave-speed parameter.

In [4]:
uu, vv = grid.register_gridfunctions(["uu", "vv"], group="EVOL")
wave_speed = par.register_CodeParameter(
    "REAL", "tutorial_overview", "overview_wave_speed", 1.0
)
energy_density = sp.Rational(1, 2) * (vv**2 + wave_speed**2 * uu**2)
assignment = ccg.c_codegen(
    energy_density,
    "energy_density",
    include_braces=False,
    verbose=False,
)
print(assignment)

energy_density = (1.0/2.0)*((overview_wave_speed)*(overview_wave_speed))*((uu)*(uu)) + (1.0/2.0)*((vv)*(vv));



## Register and Inspect the C Function

The prototype is the short promise of the generated function. Check that the
name, input values, and output pointer match the energy calculation.

In [5]:
register_CFunction(
    name="overview_energy_density",
    desc="Compute a compact wave-energy diagnostic.",
    cfunc_type="void",
    params=(
        "const double uu, const double vv, "
        "const double wave_speed, double *energy"
    ),
    body=(
        "*energy = 0.5*(vv*vv + wave_speed*wave_speed*uu*uu);"
        "\nreturn;"
    ),
)
prototype = CFunction_dict["overview_energy_density"].function_prototype
print("registered keys:", sorted(CFunction_dict))
print(prototype)

registered keys: ['overview_energy_density']
void overview_energy_density(const double uu, const double vv, const double wave_speed, double *energy);


## Validation Check

A valid miniature pipeline must preserve the symbolic ingredients in the C
assignment and register the function under the expected prototype.

In [6]:
expected_assignment_terms = [
    "energy_density =",
    "overview_wave_speed",
    "uu",
    "vv",
]
missing_terms = [
    term for term in expected_assignment_terms if term not in assignment
]
if missing_terms:
    raise RuntimeError(f"Missing generated-assignment terms: {missing_terms}")

stored_function = CFunction_dict.get("overview_energy_density")
if stored_function is None:
    raise RuntimeError("overview_energy_density was not stored.")
if stored_function.name != "overview_energy_density":
    raise RuntimeError(f"Unexpected function name: {stored_function.name}")
if stored_function.cfunc_type != "void":
    raise RuntimeError(f"Unexpected function type: {stored_function.cfunc_type}")
for expected_param in ["uu", "vv", "wave_speed", "energy"]:
    if expected_param not in stored_function.params:
        raise RuntimeError(f"Missing parameter {expected_param}")

print("validated assignment terms:", expected_assignment_terms)
print("validated stored function:", stored_function.name)
print("validated function parameters:", stored_function.params)

validated assignment terms: ['energy_density =', 'overview_wave_speed', 'uu', 'vv']
validated stored function: overview_energy_density
validated function parameters: const double uu, const double vv, const double wave_speed, double *energy


## Learning Check

After running the validation cell, identify which names are stored by NRPy and
which name exists only as a local Python variable in this notebook.

## Continue to Parameters

- [Parameters](params.ipynb)
- [Indexed Expressions](indexedexp.ipynb)
- [Gridfunctions and Grid Metadata](grid.ipynb)
- [C Code Generation](c_codegen.ipynb)
- [C Function Registration](c_function.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)